## Seq2Seq and Encoder - Decoder Models with LSTMs

In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam
from torch.utils.data import TensorDataset, DataLoader
import lightning as L

## Create the Datasets that will be used for training Encoder - Decoder model

We will translate **Let's go** and **to go** into spanish. **Let's go** should translate to **vamos<EOS>** and **to go** should translate to **ir<EOS>**

In [4]:
# We create a dictionary that maps vocabulary tokens to id numbers
english_token_to_id = {'lets':0,
                       'to': 1,
                       'go': 2,
                       '<EOS': 3
}

# Next we create a dictionary that maps the ids to tokens
english_id_to_token = dict(map(reversed, english_token_to_id.items()))

spanish_token_to_id = {'ir': 0,
                       'vamos': 1,
                       'y': 2,
                       '<EOS>': 3}

spanish_id_to_token = dict(map(reversed, spanish_token_to_id.items()))

inputs = torch.tensor([[english_token_to_id["lets"],
                        english_token_to_id["go"]],
                        
                        [english_token_to_id["to"],
                        english_token_to_id["go"]]])

labels = torch.tensor([[spanish_token_to_id["vamos"],
                        spanish_token_to_id["<EOS>"]],
                        
                        [spanish_token_to_id["ir"],
                        spanish_token_to_id["<EOS>"]]])


dataset = TensorDataset(inputs, labels)
dataloader = DataLoader(dataset)

## Building and Training a Seq2Seq Encoder - Decoder Model From Scratch

In [5]:
class seq2seq(L.LightningModule):
    def __init__(self, max_len=2):
        super().__init__()

        self.max_output_length = max_len
        L.seed_everything(seed=42)

        # Encoding 
        self.encoder_we = nn.Embedding(num_embeddings=4, # number of words as inputs
                                       embedding_dim=2) # numbers per embedding
        
        self.encoder_lstm = nn.LSTM(input_size=2, # number of inputs (2 numbers per word)
                                    hidden_size=2, # number of outputs (2 per word per layer) (its the number of outputs of the short-term memory)
                                    num_layers=2) # number of LSTMs to stack. If there are 2  layers,
                                                    # then the short term memory from the first layer is used as input to the second layer
        
        # Decoding
        self.decoder_we = nn.Embedding(num_embeddings=4,
                                       embedding_dim=2)
        
        self.decoder_lstm = nn.LSTM(input_size=2,
                                    hidden_size=2,
                                    num_layers=2)
        
        self.output_fc = nn.Linear(in_features=2, # number of outputs per LSTM
                                   out_features=4) # number of words in the output vocabulary
        

        # Training
        self.loss = nn.CrossEntropyLoss()

    def forward(self, input, output=None):
        # Encoding
        encoder_embeddings = self.encoder_we(input)
        encoder_lstm_output, (encoder_lstm_hidden, encoder_lstm_cell) = self.encoder_lstm(encoder_embeddings)
        # encoder_lstm_output is the short-term memory at every timestep (we dont use that)
        # encoder_lstm_hidden is the final short-term-memory
        # encoder_lstm_cell is the final long-term memory

        # Decoding
        # We initialize the decoder with <EOS> token
        decoder_token_id = torch.tensor([spanish_token_to_id['<EOS>']])
        decoder_embeddings = self.decoder_we(decoder_token_id)

        # We initialize the decoder LSTM with the encoders final short and long-term memory instead of zeros. 
        # Thats how the decoder knows what the input sentence was
        decoder_lstm_output, (decoder_lstm_hidden, decoder_lstm_cell) = self.decoder_lstm(decoder_embeddings,
                                                                                          (encoder_lstm_hidden,
                                                                                          encoder_lstm_cell))
        
        #Takes the decoder LSTM's output (2 numbers) and passes it through the linear layer to get 4 scores — one per vocabulary word. 
        # These are raw logits, not probabilities yet.
        output_values = self.output_fc(decoder_lstm_output)
        outputs = output_values # first tensor, acts as the starting container for the for loop

        predicted_id = torch.tensor([torch.argmax(output_values)]) # torch.argmax() finds the index of the highest score that's the model's predicted token ID for this step.
        predicted_ids = predicted_id                               # Wraps it in a tensor so it can be fed back into the embedding layer next iteration.
        # L first tensor, acts as the starting container for the for loop                                                    
    

        for i in range(1, self.max_output_length): # we start from 1 because we already produced the first token above
            if (output == None):
                if (predicted_id == spanish_token_to_id['<EOS>']): # if last predicted token was <EOS> we are done
                    break
                decoder_embeddings = self.decoder_we(predicted_id) # we embed the model's own last prediction and feed it in for the next step
            else:
                decoder_embeddings = self.decoder_we(torch.tensor([output[i-1]])) # We are always feeding one step behind, because the decoder's job is to predict the next token given the current one.
            
            # This time we pass decoders final short and long-term memory as it has done a pass already
            decoder_lstm_output, (decoder_lstm_hidden, decoder_lstm_cell) = self.decoder_lstm(decoder_embeddings,
                                                                                              (decoder_lstm_hidden,
                                                                                              decoder_lstm_cell))
            output_values = self.output_fc(decoder_lstm_output)
            outputs = torch.cat((outputs, output_values), 0)
            predicted_id = torch.tensor([torch.argmax(output_values)])
            predicted_ids = torch.cat((predicted_ids, predicted_id))
        return (outputs)
    
    def configure_optimizers(self):
        return Adam(self.parameters(), lr=0.1)
    
    def training_step(self, batch, batch_idx):
        input_tokens, labels = batch
        output = self.forward(input_tokens[0], labels[0])
        loss = self.loss(output, labels[0])
        return loss

In [6]:
model = seq2seq()

inputs = torch.tensor([
    english_token_to_id['lets'],
    english_token_to_id['go']
])

outputs = model.forward(input=inputs, output=None)

print("Translated text...")
predicted_ids = torch.argmax(outputs, dim=1)
for id in predicted_ids:
    print("\t", spanish_id_to_token[id.item()])

Seed set to 42


Translated text...
	 y
	 y


In [7]:
trainer = L.Trainer(max_epochs=80, accelerator='cpu')
trainer.fit(model, train_dataloaders=dataloader)

GPU available: True (mps), used: False
TPU available: False, using: 0 TPU cores
/Users/odysseasgeorgiades/Desktop/Neural Networks/.venv/lib/python3.14/site-packages/lightning/pytorch/trainer/setup.py:175: GPU available but not used. You can set it by doing `Trainer(accelerator='gpu')`.
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.

  | Name         | Type             | Params | Mode  | FLOPs
------------------------------------------------------------------
0 | encoder_we   | Embedding        | 8      | train | 0    
1 | encoder_lstm | LSTM             | 96     | train | 0    
2 | decoder_we   | Embeddin

Epoch 79: 100%|██████████| 2/2 [00:00<00:00, 456.92it/s, v_num=2]

`Trainer.fit` stopped: `max_epochs=80` reached.


Epoch 79: 100%|██████████| 2/2 [00:00<00:00, 280.81it/s, v_num=2]


In [9]:
# Translate "Let's go" to "vamos <EOS>"
inputs = torch.tensor([
    english_token_to_id['lets'],
    english_token_to_id['go']
])

outputs = model.forward(input=inputs, output=None)

print("Translated text...")
predicted_ids = torch.argmax(outputs, dim=1)
for id in predicted_ids:
    print("\t", spanish_id_to_token[id.item()])

Translated text...
	 vamos
	 <EOS>


In [10]:
# Translate "to go" to "ir <EOS>"
inputs = torch.tensor([
    english_token_to_id['to'],
    english_token_to_id['go']
])

outputs = model.forward(input=inputs, output=None)

print("Translated text...")
predicted_ids = torch.argmax(outputs, dim=1)
for id in predicted_ids:
    print("\t", spanish_id_to_token[id.item()])

Translated text...
	 ir
	 <EOS>


In [11]:
# Count the number of parameters we had to train 
total_trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print("Total number of trainable parameters: ", total_trainable_params)

Total number of trainable parameters:  220


## Saving and Loading the trained model weights

In [13]:
# We save the weights
trainer.save_checkpoint("seq2seq_en2en_220_trained.ckpt")

`weights_only` was not set, defaulting to `False`.


In [14]:
# We create a new model and load in the saved weights
new_model = seq2seq.load_from_checkpoint("seq2seq_en2en_220_trained.ckpt")

# Translate "Let's go" to "vamos <EOS>"
inputs = torch.tensor([
    english_token_to_id['lets'],
    english_token_to_id['go']
])

outputs = model.forward(input=inputs, output=None)

print("Translated text...")
predicted_ids = torch.argmax(outputs, dim=1)
for id in predicted_ids:
    print("\t", spanish_id_to_token[id.item()])

Seed set to 42


Translated text...
	 vamos
	 <EOS>
